# The Ghost Eigenvector

## Congratss It's the garbage finale!

Sit down. I want to tell you about the best result I have ever seen a machine learning pipeline produce, and I want you to understand, deeply, in your bones, why it is a lie.

You are about to run a PCA on the Wine Quality dataset, reduce eleven chemistry columns down to two friendly little numbers, and watch the first component alone claim to explain **over 99% of the variance**. Ninety nine percent. From two numbers. That is not a dimensionality reduction, that is a magic trick, and like all magic tricks it works by getting you to look at the wrong hand.

Then you will plot those two components, expecting to see nice clean clusters of good wine and bad wine sorted out like laundry. Instead you get a blob. A tight, smug, uninformative little blob, with good and bad wine sitting on top of each other like they're sharing a booth. Feed that blob to a classifier and it will perform barely better than a coin flip that read the label first.

Somewhere in this notebook, PCA is telling the truth about the wrong question. Your job is to find out which question it actually answered.

### Run It

Run every cell top to bottom exactly as written and watch the explained variance number lie to your face.

#### Step 1: Load and inspect

Nothing clever here. We read a CSV. If you mess this part up I genuinely don't know what to tell you.

In [ ]:
# Load the raw dataset and take a first look
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

RANDOM_STATE = 42

df = pd.read_csv("dataset/WineQT.csv")
df.head()

#### Step 2: Clean up and define the target

We drop the row identifier (it's not a chemical property, it's bookkeeping), and turn the quality
score into a binary "good wine / not good wine" label so we have a classification problem worth
solving downstream.

In [ ]:
# Drop the identifier column and build a binary quality label
df = df.drop(columns=["Id"])
df["quality_label"] = (df["quality"] >= 6).astype(int)

feature_cols = [c for c in df.columns if c not in ("quality", "quality_label")]
print("Feature columns:", feature_cols)
print("Class balance:")
print(df["quality_label"].value_counts(normalize=True))

#### Step 3: Train/test split

Standard practice, held out test set, stratified so both classes show up in both splits.

In [ ]:
# Split features and target into train and test sets
X = df[feature_cols].to_numpy(dtype=float)
y = df["quality_label"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

#### Step 4: Scale features before PCA

Eleven columns, eleven totally different units. Density lives in a tiny window around 1.0, total
sulfur dioxide wanders around in the hundreds. Before PCA gets anywhere near this data, every
feature needs to be brought onto a comparable scale so one column doesn't just win by having
bigger numbers.

In [ ]:
# Rescale each feature so no single column dominates purely by magnitude
def scale_features(X_train, X_test):
    means = X_train.mean(axis=0)
    stds = X_train.std(axis=0)
    return (X_train - means) / stds, (X_test - means) / stds, stds

X_train_scaled, X_test_scaled, feature_stds = scale_features(X_train, X_test)
print("Post-scaling per-feature std (train):")
print(np.round(X_train_scaled.std(axis=0), 3))

#### Step 5: PCA, implemented by hand

We are not importing this one from sklearn. You should see the eigendecomposition happen with your
own eyes at least once in your life. Two functions: one builds the matrix PCA eigendecomposes,
the other does the eigendecomposition and keeps the top components.

In [ ]:
# Build the matrix that PCA will eigendecompose
def compute_covariance_matrix(X):
    n_samples = X.shape[0]
    X_centered = X - X.mean(axis=0)
    cov = (X_centered.T @ X_centered) / n_samples
    return cov


# Eigendecompose that matrix and keep the top n_components directions
def fit_pca(X, n_components=2):
    cov = compute_covariance_matrix(X)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)

    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    components = eigenvectors[:, :n_components]
    explained_variance_ratio = eigenvalues[:n_components] / eigenvalues.sum()
    return components, explained_variance_ratio

#### Step 6: Project the data down to 2D

This is the moment of truth. Watch the explained variance ratio.

In [ ]:
# Fit PCA on the training set and project both splits into 2D
components, explained_variance_ratio = fit_pca(X_train_scaled, n_components=2)

print("Explained variance ratio:", explained_variance_ratio)
print("Total explained by 2 components: {:.2%}".format(explained_variance_ratio.sum()))

X_train_pca = X_train_scaled @ components
X_test_pca = X_test_scaled @ components

#### Step 7: Two classes, two colors

In [ ]:
# Visualize the 2D projection, colored by wine quality label
plt.figure(figsize=(6, 5))
for label, color, name in [(0, "tab:red", "lower quality"), (1, "tab:blue", "higher quality")]:
    mask = y_train == label
    plt.scatter(X_train_pca[mask, 0], X_train_pca[mask, 1], s=12, alpha=0.6, c=color, label=name)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("2D PCA projection of wine chemistry")
plt.legend()
plt.tight_layout()
plt.show()

#### Step 8: Downstream classifier

Train something simple on the 2D projection and see how it does. If PCA actually found the
directions that separate good wine from bad wine, this should not be hard.

In [ ]:
# Train a classifier on the 2D PCA-reduced features and evaluate it
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_pca, y_train)
preds = clf.predict(X_test_pca)

print("Test accuracy on PCA(2) features:", round(accuracy_score(y_test, preds), 4))
print("Majority-class baseline:", round(max(y_test.mean(), 1 - y_test.mean()), 4))
print(classification_report(y_test, preds, digits=3))

### The Math Clue

Alright. Everyone thinks they understand PCA. Almost nobody does. Let's fix that.

PCA promises you one thing: find the directions along which your data varies the most.
Mathematically, that means eigendecomposing the covariance matrix:

$$
\Sigma = \frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})(x_i - \bar{x})^\top
$$

Look at that formula. Really look at it. Every term is a deviation from the mean, $\bar{x}$. Not
the raw point, the deviation. PCA was never about your data, it was always about how your data
moves around its own center. Variance is a statement about spread, and spread is only measurable
relative to a center.

Now, ask yourselves what happens if you hand PCA the raw, uncentered data? 

### Your Mission

Somewhere in the cells above, PCA was a little cooked. Find. Fix. 

### Submission

1. Fork the repository.
2. Fix the bug in this notebook's case folder.
3. Open a pull request against the original repo.
4. A hooman confirms your fix actually resolves the symptom described above.
5. Once confirmed, you are awarded your title. See below, if you dare look ahead.

<details>
<summary><b>Mock Ceremony Title (click to reveal, if you think you've earned it)</b></summary>

### Supreme Archon of the Centered Universe, Grand Curator of Eigenspace, First of Their Name to Ever Subtract a Mean Correctly

"Do you understand what you've done. bleeeeeep the answer and you *saw through it*. Most people go their
entire careers running PCA on uncentered data and never once notice the ghost. You noticed the
ghost. Rise, Archon. Try not to let it go to your head, there's a solid chance it'll just get
bigger."

It was a pleasure having you my guy 
</details>